# 02 Prepare Reasoning Prompts

Load only the assigned contiguous raw shard, prepare clean reasoning prompt records for that shard, and save `reasoning_prompts_shardX.jsonl`. No modulo sharding is used here.


This cell imports dependencies and configures the assigned shard input, prompt output path, and optional unassigned-shard deletion behavior.

In [ ]:
# merging for testing
import pandas as pd

cleaned_path = r"E:\PFE_CoT_outputs\data\interim\cleaned_cicmaldroid_shard2.csv"
summary_path = r"E:\PFE_CoT_outputs\data\interim\json_behavior_summary_shard2.csv"

cleaned = pd.read_csv(cleaned_path, low_memory=False)
summary = pd.read_csv(summary_path, low_memory=False)

print("Cleaned cols:", cleaned.columns.tolist())
print("Summary cols:", summary.columns.tolist())

In [ ]:
# quick check 
# Patch: merge json_behavior_summary into cleaned CSV
summary_cols_to_add = [col for col in summary.columns 
                       if col not in cleaned.columns and col not in ('sample_id', 'original_id')]

cleaned_patched = cleaned.merge(
    summary[['original_id'] + summary_cols_to_add],
    on='original_id',
    how='left'
)

print("Rows before:", len(cleaned), "| Rows after:", len(cleaned_patched))
print("json_behavior_summary non-null:", cleaned_patched["json_behavior_summary"].notna().sum())
print("Sample value:")
print(cleaned_patched["json_behavior_summary"].dropna().iloc[0])

# Save back to the same path
cleaned_patched.to_csv(cleaned_path, index=False)
print("Patched CSV saved.")

In [ ]:
from pathlib import Path
import json
import math
import re

import pandas as pd
from IPython.display import display

# Each machine edits this value and processes only its assigned shard.
ASSIGNED_SHARD_ID = 2  # 1 or 2
DELETE_UNASSIGNED_SHARD = False
NUM_SHARDS = 2

PROJECT_ROOT = Path("..").resolve() if Path.cwd().name == "notebooks" else Path(".").resolve()
# CHANGED: generated/intermediate outputs stay on E: to avoid filling the C: drive.
OUTPUT_ROOT = Path(r"E:\PFE_CoT_outputs")
INTERIM_DIR = OUTPUT_ROOT / "data" / "interim"
PROMPTS_DIR = PROJECT_ROOT / "prompts"

RAW_SHARD_PATHS = {
    1: INTERIM_DIR / "raw_shard1.csv",
    2: INTERIM_DIR / "raw_shard2.csv",
}

SYSTEM_PROMPT_PATH = PROMPTS_DIR / "cicmaldroid_system_prompt.txt"
USER_PROMPT_TEMPLATE_PATH = PROMPTS_DIR / "cicmaldroid_user_prompt_template.txt"

SOURCE_DATASET = "CICMalDroid2020"
MAX_SAMPLES = 100  # small number for testing, None for the full assigned shard
MAX_EVIDENCE_ITEMS = None  # Set to a number like 120 only if prompts exceed model context.

if ASSIGNED_SHARD_ID not in RAW_SHARD_PATHS:
    raise ValueError("ASSIGNED_SHARD_ID must be 1 or 2.")

ASSIGNED_RAW_SHARD_PATH = RAW_SHARD_PATHS[ASSIGNED_SHARD_ID]
# CHANGED: prefer the cleaned shard because notebook 01 adds compact JSON behavior evidence there.
CLEANED_INPUT_PATH = INTERIM_DIR / f"cleaned_cicmaldroid_shard{ASSIGNED_SHARD_ID}.csv"
PROMPT_INPUT_PATH = CLEANED_INPUT_PATH if CLEANED_INPUT_PATH.exists() else ASSIGNED_RAW_SHARD_PATH
PROMPT_OUTPUT_PATH = INTERIM_DIR / f"reasoning_prompts_shard{ASSIGNED_SHARD_ID}.jsonl"

PROMPTS_DIR.mkdir(parents=True, exist_ok=True)
INTERIM_DIR.mkdir(parents=True, exist_ok=True)

opposite_shard_id = 2 if ASSIGNED_SHARD_ID == 1 else 1
opposite_path = RAW_SHARD_PATHS[opposite_shard_id]
if DELETE_UNASSIGNED_SHARD and opposite_path.exists():
    opposite_path.unlink()
    print(f"Kept assigned raw shard {ASSIGNED_SHARD_ID}: {ASSIGNED_RAW_SHARD_PATH}")
    print(f"Removed unassigned intermediate raw shard {opposite_shard_id}: {opposite_path}")
elif DELETE_UNASSIGNED_SHARD:
    print(f"Kept assigned raw shard {ASSIGNED_SHARD_ID}: {ASSIGNED_RAW_SHARD_PATH}")
    print(f"Unassigned intermediate raw shard {opposite_shard_id} is already absent: {opposite_path}")
else:
    print("DELETE_UNASSIGNED_SHARD is False; keeping both raw shard CSV files.")

if not PROMPT_INPUT_PATH.exists():
    raise FileNotFoundError(f"Missing assigned shard input. Run notebook 01 with ASSIGNED_SHARD_ID={ASSIGNED_SHARD_ID}. Expected cleaned or raw shard at: {PROMPT_INPUT_PATH}")

print("Prompt input shard:", PROMPT_INPUT_PATH)
print("Prompt JSONL output:", PROMPT_OUTPUT_PATH)
print("Prompts folder:", PROMPTS_DIR)


This cell defines the system and user prompt templates and saves them under `prompts/` for reproducibility.

In [ ]:
SYSTEM_PROMPT = SYSTEM_PROMPT = """You are a malware analysis assistant specializing in Android applications.

Your task is to analyze behavioral evidence extracted from an Android application and produce an evidence-driven structured analysis.

Rules:

- Base your reasoning exclusively on the evidence provided. Do not infer, assume, or invent any permissions, API calls, network indicators, package names, families, or behaviors not explicitly present in the evidence.

- Do not mention that you are an AI, that this is a dataset, or that labels exist.

- The Reasoning paragraph must analyze the evidence first and end with a conclusion that supports the final verdict. A reader who sees only the Reasoning, without the Verdict line, must be able to predict the correct label from the argument alone.

- Explicitly compare observed behavior against what legitimate Android applications typically require. Flag anomalies, excessive permissions, or suspicious API combinations with clear analytical justification.

- Do not use hedge language such as "could be", "may indicate", "might suggest", or "could be benign or malicious". Every observation must be tied to a directional analytical conclusion.

- Do not state the verdict word explicitly inside the Reasoning paragraph. The argument must imply it through logic, not assert it by name.

- The Verdict must be exactly one of: Benign, Adware, Banking, SMS, Riskware

- Do not add confidence scores, qualifiers, or explanation to the Verdict line.

- Write the Reasoning as a single concise paragraph of 4 to 6 sentences maximum.

Output format (follow exactly):

Reasoning:

<one concise paragraph of 4-6 sentences — analytical argument that makes the verdict logically inevitable>

Verdict:

<one label from: Benign, Adware, Banking, SMS, Riskware>"""

USER_PROMPT_TEMPLATE = """Analyze the behavioral evidence of the following Android application.

Behavioral evidence:
{evidence}

When analyzing, explicitly address:
1. Whether the permissions and API calls are proportionate to what the application's apparent purpose would legitimately require
2. Whether any combinations, frequencies, or patterns are characteristic of a specific threat category such as financial data interception, premium service abuse, intrusive advertising, or elevated system privilege use
3. What the overall behavioral profile implies about the application's intent, stated as a conclusion not a possibility

Base your analysis solely on the evidence above. Do not restate the evidence verbatim — summarize what the behavior implies analytically."""
# Store the exact prompt text used for reproducibility.
SYSTEM_PROMPT_PATH.write_text(SYSTEM_PROMPT + "\n", encoding="utf-8")
USER_PROMPT_TEMPLATE_PATH.write_text(USER_PROMPT_TEMPLATE + "\n", encoding="utf-8")

print("System prompt saved:", SYSTEM_PROMPT_PATH)
print("User prompt template saved:", USER_PROMPT_TEMPLATE_PATH)


This cell loads the assigned prompt input, preferring the cleaned shard with JSON behavior evidence and falling back to the raw shard only if needed.

In [ ]:
df = pd.read_csv(PROMPT_INPUT_PATH, low_memory=False)
if "sample_id" not in df.columns:
    raise KeyError("Assigned raw shard must contain the temporary sample_id from notebook 01.")
if "original_id" not in df.columns:
    raise KeyError("Assigned raw shard must contain original_id from notebook 01.")

df["sample_id"] = df["sample_id"].astype(int)
df["original_id"] = df["original_id"].astype(str)
df = df.sort_values("sample_id").reset_index(drop=True)
if MAX_SAMPLES is not None:
    df = df.head(MAX_SAMPLES).copy()

print("Rows loaded for assigned shard:", len(df))
print("sample_id range:", int(df["sample_id"].min()), "to", int(df["sample_id"].max()))
print("Unique original_id values:", df["original_id"].nunique())
display(df[["sample_id", "original_id"]].head())


This cell detects label/category/family metadata, preserves `original_id`, and selects behavioral evidence columns for prompting.

In [ ]:
def normalize_name(name):
    return re.sub(r"[^a-z0-9]+", "", str(name).lower())


def find_first_column(columns, candidates):
    normalized = {normalize_name(column): column for column in columns}
    for candidate in candidates:
        key = normalize_name(candidate)
        if key in normalized:
            return normalized[key]
    for column in columns:
        col_key = normalize_name(column)
        if any(normalize_name(candidate) in col_key for candidate in candidates):
            return column
    return None


def is_missing(value):
    if value is None:
        return True
    try:
        if pd.isna(value):
            return True
    except TypeError:
        pass
    if isinstance(value, str) and value.strip() == "":
        return True
    return False


def clean_value(value):
    if is_missing(value):
        return "unknown"
    if isinstance(value, float) and value.is_integer():
        return str(int(value))
    return str(value).strip()


IDENTIFIER_CANDIDATES = [
    "original_id", "dataset_key", "key", "sample_key", "json_filename", "json_file",
    "sha256", "sha1", "md5", "hash", "apk_hash", "apk", "apk_name", "file",
    "filename", "file_name", "path", "file_path", "app", "app_name", "package",
    "package_name", "pkg_name", "name", "raw_identifier",
]
LABEL_CANDIDATES = [
    "true_label", "label", "class", "classification", "malware", "is_malware",
    "benign_malware", "type", "verdict",
]
CATEGORY_CANDIDATES = [
    "category", "malware_category", "malware_type", "class", "classification", "type",
]
FAMILY_CANDIDATES = [
    "family", "malware_family", "family_name", "malware_name", "variant", "subclass",
]

label_col = find_first_column(df.columns, LABEL_CANDIDATES)
category_col = find_first_column(df.columns, CATEGORY_CANDIDATES)
family_col = find_first_column(df.columns, FAMILY_CANDIDATES)
identifier_col = "raw_identifier" if "raw_identifier" in df.columns else find_first_column(df.columns, IDENTIFIER_CANDIDATES)
true_label_source = label_col or category_col or family_col

df_prompt = df.copy()
df_prompt["source_dataset"] = SOURCE_DATASET
df_prompt["original_id"] = df_prompt["original_id"].map(clean_value)
df_prompt["true_label"] = df_prompt[true_label_source].map(clean_value) if true_label_source else "unknown"
df_prompt["category"] = df_prompt[category_col].map(clean_value) if category_col else df_prompt["true_label"]
df_prompt["family"] = df_prompt[family_col].map(clean_value) if family_col else "unknown"

metadata_columns = {
    "sample_id", "original_id", "source_dataset", "true_label", "category", "family",
    "raw_identifier", "source_file", "source_row_order",
    "json_file", "json_path", "label_source_file", "json_sha256", "json_md5",
    "json_process_name", "json_package", "json_parse_error",
}
metadata_columns.update(column for column in [label_col, category_col, family_col, identifier_col] if column)
identifier_exact_keys = {
    "name", "app", "appname", "apk", "apkname", "file", "filename", "filepath",
    "path", "jsonpath", "jsonfile", "rawjsonpath", "packagename", "package", "pkgname",
    "processname", "rawidentifier", "originalid", "sha256", "md5",
}
metadata_keywords = [
    "label", "class", "category", "family", "verdict", "hash", "sha", "md5",
    "identifier", "sourcefile", "labelsourcefile",
]

preferred_evidence_columns = [
    column for column in ["json_behavior_summary"]
    if column in df_prompt.columns and not df_prompt[column].isna().all()
]

evidence_columns = []
for column in df_prompt.columns:
    key = normalize_name(column)
    if column in metadata_columns or key in identifier_exact_keys:
        continue
    if any(keyword in key for keyword in metadata_keywords):
        continue
    if df_prompt[column].isna().all():
        continue
    evidence_columns.append(column)

# CHANGED: use the compact JSON behavior summary as the primary prompt evidence when it exists.
if preferred_evidence_columns:
    evidence_columns = preferred_evidence_columns

print("Detected label column:", label_col)
print("Detected category column:", category_col)
print("Detected family column:", family_col)
print("Detected identifier column kept for traceability only:", identifier_col)
print("Evidence columns selected:", len(evidence_columns))
print(evidence_columns[:100])
print("\nCategory distribution for assigned shard:")
print(df_prompt["category"].value_counts(dropna=False).head(30))


This cell builds evidence text and prompt records while keeping both `sample_id` and `original_id` in every JSONL record.

In [ ]:
def include_feature_value(value):
    if is_missing(value):
        return False
    if isinstance(value, (int, float)) and not isinstance(value, bool):
        return not math.isclose(float(value), 0.0)
    text = str(value).strip().lower()
    return text not in {"0", "0.0", "false", "no", "none", "nan", "unknown", "?"}


def build_evidence(row):
    parts = []
    for column in evidence_columns:
        value = row[column]
        if not include_feature_value(value):
            continue
        parts.append(f"{column}: {clean_value(value)}")

    if MAX_EVIDENCE_ITEMS is not None and len(parts) > MAX_EVIDENCE_ITEMS:
        parts = parts[:MAX_EVIDENCE_ITEMS] + [f"truncated_evidence_fields: {len(parts) - MAX_EVIDENCE_ITEMS} additional non-empty fields not shown"]

    if not parts:
        return "unknown"
    return "\n".join(parts)


def prompt_record(row):
    evidence = build_evidence(row)
    user_prompt = USER_PROMPT_TEMPLATE.format(evidence=evidence)

    return {
        "sample_id": int(row["sample_id"]),
        "original_id": clean_value(row.get("original_id", "unknown")),
        "source_dataset": SOURCE_DATASET,
        "true_label": clean_value(row.get("true_label", "unknown")),
        "category": clean_value(row.get("category", "unknown")),
        "family": clean_value(row.get("family", "unknown")),
        "evidence": evidence,
        "system_prompt": SYSTEM_PROMPT,
        "user_prompt": user_prompt,
        "prompt": SYSTEM_PROMPT + "\n\n" + user_prompt,
    }


def write_jsonl(path, records):
    with open(path, "w", encoding="utf-8") as output_handle:
        for record in records:
            output_handle.write(json.dumps(record, ensure_ascii=False) + "\n")


In [ ]:
# Optional: balance categories for testing
BALANCE_SAMPLES = True
SAMPLES_PER_CATEGORY = 15  # adjust as needed

if BALANCE_SAMPLES:
    df_prompt = (
        df_prompt
        .groupby("category", group_keys=False)
        .apply(lambda g: g.sample(min(len(g), SAMPLES_PER_CATEGORY), random_state=42))
        .reset_index(drop=True)
    )
    print("Balanced category distribution:")
    print(df_prompt["category"].value_counts())

This cell writes the assigned shard prompt JSONL file and prints a category distribution for that shard.

In [ ]:
json_summary_filter_count = 0
if "json_behavior_summary" in df_prompt.columns:
    json_summary_text = df_prompt["json_behavior_summary"].astype("string")
    invalid_json_summary_mask = json_summary_text.isna() | json_summary_text.str.startswith("json_parse_error", na=False)
    json_summary_filter_count = int(invalid_json_summary_mask.sum())
    df_prompt = df_prompt.loc[~invalid_json_summary_mask].reset_index(drop=True)
else:
    print("json_behavior_summary column not found; no JSON parse-error prompt rows filtered.")
print("Filtered rows with missing or parse-error JSON behavior summary:", json_summary_filter_count)

records = [prompt_record(row) for _, row in df_prompt.iterrows()]
write_jsonl(PROMPT_OUTPUT_PATH, records)

print(f"Assigned shard {ASSIGNED_SHARD_ID}/{NUM_SHARDS}: {len(records)} prompt records -> {PROMPT_OUTPUT_PATH}")
if records:
    shard_df = pd.DataFrame(records)
    print("Category distribution:")
    print(shard_df["category"].value_counts(dropna=False).head(30))


This cell previews a small number of generated prompt records without printing the full prompt text.

In [ ]:
preview_df = pd.DataFrame(records[:2])
if not preview_df.empty:
    preview_df["evidence"] = preview_df["evidence"].str.slice(0, 500) + "..."
    preview_df["user_prompt"] = preview_df["user_prompt"].str.slice(0, 500) + "..."
    display(preview_df[["sample_id", "original_id", "true_label", "category", "family", "evidence", "user_prompt"]])
